In [ ]:
import sys
from pyprojroot import here

# Ritorna il percorso assoluto della root del progetto
PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)
sys.dont_write_bytecode = True

print(f"La root del progetto è: {PROJECT_ROOT}")

In [ ]:
from paths import INV_PATH, TP2_PATH, MODEL_PATH, COLUMN_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP2_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + COLUMN_PATH

model_name = "column.blend"

In [ ]:
# Create needed folders
import os

lst_folders = ["figures", "files", "models", "column_parabolic_inverse", "trainPOD"]

for name in lst_folders:
    os.makedirs(os.path.join(ABS_PATH, name), exist_ok=True)

Import delle librerie

In [ ]:
import torch
import pandas as pd

from modelaquisition.bl2pina import Blend2Pina
from modelaquisition.bl2msh import Blend2Mesh
from modelaquisition.msh2xdmf import Msh2Xdmf

Fissiamo precisione doppia

In [ ]:
torch.set_default_dtype(torch.float64)

## Creazione dei dati per il problema inverso

Fissiamo i punti in cui sono installati i sensori

In [ ]:
column = Blend2Pina(LOAD_MODEL + model_name)

Acquisizione dei punti al contrno

In [ ]:
num_points = 1_000

surface = column.boundary(time_interval=[0, 1])
points = surface.sample(num_points)

Fissimao i parametri e collezioniamo i dati simulati

In [ ]:
par_lambda = .1
par_alpha = .2
par_beta = .5

u1 = torch.exp(par_lambda*points.extract('t')) + par_alpha*points.extract('x') + par_beta*points.extract('y') + points.extract('z')
u2 = torch.exp(par_lambda*points.extract('t')) + par_alpha*(points.extract('x')**2) + par_beta*(points.extract('y')**2) + points.extract('z')**2

Creazione file .csv per conservare i punti al contorno

In [ ]:
total_info = torch.concat(
    [points.tensor, u1, u2],
    1
)

df = pd.DataFrame(
    data=total_info.numpy(),
    columns=["x", "y", "z", "t", "u1", "u2"]
)

df.to_csv("./files/data.csv", sep=";")

## Creazione mesh e xdmf

In [ ]:
column_msh = Blend2Mesh(LOAD_MODEL + model_name, "column")

In [ ]:
column_msh.create_single_meshes(len_msh=0.01)

In [ ]:
column_xdmf = Msh2Xdmf("column.msh", "column")
column_xdmf.to_xdmf(num_refine=3)
column_xdmf.to_xdmf()